# Step 4 — Matching ONTD stops to classified OSM stations

Matches every stop in `data/bahnhoefe_stops_sorted.csv` (Open Night Train
Database export) to the best candidate in
`data/step3b_output_osm_stations_classified.csv` (OSM stations classified
by an earlier pipeline step) and writes the result to
`step4_MatchingONTDtoOSM.csv`. Stops for which no candidate could be found
are additionally written to `stations_notfound.csv`.

Matching primarily by station name (exact normalized match,
falling back to fuzzy matching via `rapidfuzz`). Geographic distance (and,
where available, country) is only used to pick among several OSM stations
that share the same name — coordinates in the source data are sometimes
wrong, so distance is never used to reject an otherwise good name match.


In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from unidecode import unidecode

## Configuration

Adjust these without touching the matching logic below.

In [2]:
# --- Config (paths + parameters) -------------------------------------------
DATA_DIR = Path("data")
ONTD_CSV = DATA_DIR / "bahnhoefe_stops_sorted.csv"
OSM_CSV = DATA_DIR / "step3b_output_osm_stations_classified.csv"
OUTPUT_CSV = Path("step4_MatchingONTDtoOSM.csv")
REVIEW_CSV = Path("stations_review.csv")
AMBIGUOUS_CSV = Path("stations_ambiguous.csv")

NAME_SIMILARITY_THRESHOLD = 85
MAX_MATCH_DISTANCE_KM = 10.0

## Load data

Both source files have shown up with mangled encodings when copy-pasted
around, so loading tries a few encodings before giving up.

In [3]:
def read_csv_robust(path: Path, **kwargs) -> pd.DataFrame:
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


df_ontd = read_csv_robust(ONTD_CSV, dtype={"ID": str})
df_osm = read_csv_robust(OSM_CSV, dtype={"stop_id": str, "stop_code": str})

EXPECTED_ONTD_COLUMNS = {
    "ID",
    "Name",
    "Name (Lateinisch)",
    "Name (ASCII)",
    "Länderkürzel",
    "Zeitzone",
    "Latitude",
    "Longitude",
}
EXPECTED_OSM_COLUMNS = {
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "country",
    "station_mode",
    "mode_rule",
    "stop_code",
}

missing_ontd = EXPECTED_ONTD_COLUMNS - set(df_ontd.columns)
missing_osm = EXPECTED_OSM_COLUMNS - set(df_osm.columns)
if missing_ontd or missing_osm:
    raise ValueError(
        f"Unexpected CSV columns — ONTD missing {missing_ontd or 'none'} "
        f"(got {list(df_ontd.columns)}), OSM missing {missing_osm or 'none'} "
        f"(got {list(df_osm.columns)}). This usually means an encoding was "
        f"guessed wrong."
    )

print(f"ONTD stops: {len(df_ontd)}")
print(f"OSM stations: {len(df_osm)}")

ONTD stops: 48617
OSM stations: 98936


## Normalize names and prepare candidate data

In [4]:
def normalize_raw(name) -> str:
    """Lowercase and strip non-word characters, keep the original script."""
    if not isinstance(name, str) or not name.strip():
        return ""
    folded = name.casefold()
    folded = re.sub(r"[^\w]+", " ", folded, flags=re.UNICODE)
    return folded.strip()


def normalize_translit(name) -> str:
    """Transliterate to ASCII and strip anything outside a-z/0-9."""
    if not isinstance(name, str) or not name.strip():
        return ""
    ascii_name = unidecode(name).lower()
    ascii_name = re.sub(r"[^a-z0-9]+", " ", ascii_name)
    return ascii_name.strip()


def distance_km(lat1, lon1, lat2, lon2):
    """Equirectangular approximation — accurate enough at station scale."""
    r = 6371.0088
    lat1, lon1, lat2, lon2 = (np.radians(v) for v in (lat1, lon1, lat2, lon2))
    mean_lat = (lat1 + lat2) / 2
    dx = (lon2 - lon1) * np.cos(mean_lat)
    dy = lat2 - lat1
    return r * np.sqrt(dx**2 + dy**2)

In [5]:
# Drop rows with no usable name — can't match on an empty string
df_osm["stop_name"] = df_osm["stop_name"].fillna("")
df_osm = df_osm[df_osm["stop_name"].str.strip() != ""].copy()

# Normalize names in both scripts (raw + transliterated) for the two-pass match below
df_osm["norm_raw"] = df_osm["stop_name"].map(normalize_raw).copy()
df_osm["norm_translit"] = df_osm["stop_name"].map(normalize_translit)
df_osm["country"] = df_osm["country"].fillna("").str.strip().str.upper()

# Coordinates arrive with comma decimal separators in some exports — normalize to numeric
for col in ("Latitude", "Longitude"):
    df_ontd[col] = pd.to_numeric(
        df_ontd[col].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )
for col in ("stop_lat", "stop_lon"):
    df_osm[col] = pd.to_numeric(
        df_osm[col].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )

# Same normalization pass for the ONTD side
df_ontd["norm_raw"] = df_ontd["Name"].map(normalize_raw)
df_ontd["norm_translit"] = df_ontd["Name"].map(normalize_translit)
df_ontd["Länderkürzel"] = df_ontd["Länderkürzel"].fillna("").str.strip().str.upper()

## Build the OSM name lookup

Exact matches are a dict lookup; the fuzzy fallback searches the same
grouping's keys with `rapidfuzz`.

In [6]:
# Group OSM stations by normalized name so exact lookups are O(1) dict hits;
# fuzzy fallback searches the same grouping's keys.
osm_by_name_raw = {name: group for name, group in df_osm.groupby("norm_raw")}
osm_unique_names_raw = list(osm_by_name_raw.keys())

osm_by_name_translit = {name: group for name, group in df_osm.groupby("norm_translit")}
osm_unique_names_translit = list(osm_by_name_translit.keys())

print(f"Unique normalized OSM names (raw script): {len(osm_unique_names_raw)}")
print(f"Unique normalized OSM names (transliterated): {len(osm_unique_names_translit)}")

Unique normalized OSM names (raw script): 76038
Unique normalized OSM names (transliterated): 75762


## Matching

For each ONTD stop, matched against the OSM name in its **original script
first**, transliteration only as fallback — avoids cross-script false
matches (e.g. Cyrillic vs. Latin scores ~0 in fuzzy matching).

1. `exact_raw` — exact match on the raw (script-preserving) name.
2. `fuzzy_raw` — if none, best `rapidfuzz` match among raw-script names ≥ `NAME_SIMILARITY_THRESHOLD`.
3. `exact_translit` — if none, exact match on the transliterated (`unidecode`) name.
4. `fuzzy_translit` — if none, best `rapidfuzz` match among transliterated names ≥ `NAME_SIMILARITY_THRESHOLD`.
5. `unmatched` — none of the above found anything.
6. Multiple same-name candidates: prefer matching `country`, then nearest
   by distance — never used to reject a match, only to break ties.
7. Picks farther than `MAX_MATCH_DISTANCE_KM` get `_coords_mismatch`
   appended to their `match_type`.


In [7]:
def pick_candidate(candidates: pd.DataFrame, ontd_lat, ontd_lon, ontd_country: str):
    """Narrow same-name candidates by country, then return the nearest one."""
    subset = candidates
    if ontd_country:
        country_matches = candidates[candidates["country"] == ontd_country]
        if not country_matches.empty:
            subset = country_matches
    distances = distance_km(
        ontd_lat, ontd_lon, subset["stop_lat"].to_numpy(), subset["stop_lon"].to_numpy()
    )
    best_idx = int(np.argmin(distances))
    return subset.iloc[best_idx], float(distances[best_idx])


def match_stop(ontd_row) -> dict:
    norm_raw = ontd_row["norm_raw"]
    norm_translit = ontd_row["norm_translit"]
    ontd_lat, ontd_lon = ontd_row["Latitude"], ontd_row["Longitude"]
    ontd_country = ontd_row["Länderkürzel"]

    result = {
        "ontd_id": ontd_row["ID"],
        "ontd_name": ontd_row["Name"],
        "ontd_name_ascii": ontd_row["Name (ASCII)"],
        "ontd_country": ontd_country,
        "ontd_lat": ontd_lat,
        "ontd_lon": ontd_lon,
        "match_type": "unmatched",
        "name_score": None,
        "osm_stop_id": None,
        "osm_stop_name": None,
        "osm_lat": None,
        "osm_lon": None,
        "osm_stop_code": None,
        "osm_station_mode": None,
        "distance_km": None,
        "candidate_count": 0,
    }

    # 1. Exact match on the raw (script-preserving) name
    name_score = 100.0
    match_type = "exact_raw"
    candidates = osm_by_name_raw.get(norm_raw)

    # 2. Fuzzy match on the raw name
    if candidates is None:
        best = process.extractOne(
            norm_raw,
            osm_unique_names_raw,
            scorer=fuzz.WRatio,
            score_cutoff=NAME_SIMILARITY_THRESHOLD,
        )
        if best is not None:
            best_name, name_score, _ = best
            candidates = osm_by_name_raw[best_name]
            match_type = "fuzzy_raw"

    # 3. Exact match on the transliterated name
    if candidates is None:
        match_type = "exact_translit"
        name_score = 100.0
        candidates = osm_by_name_translit.get(norm_translit)

    # 4. Fuzzy match on the transliterated name — last resort before "unmatched"
    if candidates is None:
        best = process.extractOne(
            norm_translit,
            osm_unique_names_translit,
            scorer=fuzz.WRatio,
            score_cutoff=NAME_SIMILARITY_THRESHOLD,
        )
        if best is None:
            return result
        best_name, name_score, _ = best
        candidates = osm_by_name_translit[best_name]
        match_type = "fuzzy_translit"

    result["name_score"] = round(name_score, 1)
    result["candidate_count"] = len(candidates)

    # Narrow same-name candidates by country + distance, flag if still far apart
    row, distance = pick_candidate(candidates, ontd_lat, ontd_lon, ontd_country)

    if distance > MAX_MATCH_DISTANCE_KM:
        match_type += "_coords_mismatch"

    result.update(
        {
            "match_type": match_type,
            "osm_stop_id": row["stop_id"],
            "osm_stop_name": row["stop_name"],
            "osm_lat": row["stop_lat"],
            "osm_lon": row["stop_lon"],
            "osm_stop_code": row["stop_code"],
            "osm_station_mode": row["station_mode"],
            "distance_km": round(distance, 1),
        }
    )
    return result

## Run the matching and save the output

In [8]:
# Find the best matching OSM stop for every ONTD stop
total = len(df_ontd)
results = []
for i, (_, row) in enumerate(df_ontd.iterrows(), start=1):
    results.append(match_stop(row))
    if i % 500 == 0 or i == total:
        print(f"{i}/{total} ({i / total:.0%})")

df_result = pd.DataFrame(results)
df_result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

REVIEW_COLUMNS = [
    "ontd_id",
    "ontd_name",
    "ontd_name_ascii",
    "ontd_country",
    "ontd_lat",
    "ontd_lon",
    "match_type",
    "candidate_count",
    "osm_stop_id",
    "osm_stop_name",
    "osm_lat",
    "osm_lon",
    "distance_km",
]


print(f"Wrote {len(df_result)} rows to {OUTPUT_CSV}")
df_result["match_type"].value_counts()

500/48617 (1%)
1000/48617 (2%)
1500/48617 (3%)
2000/48617 (4%)
2500/48617 (5%)
3000/48617 (6%)
3500/48617 (7%)
4000/48617 (8%)
4500/48617 (9%)
5000/48617 (10%)
5500/48617 (11%)
6000/48617 (12%)
6500/48617 (13%)
7000/48617 (14%)
7500/48617 (15%)
8000/48617 (16%)
8500/48617 (17%)
9000/48617 (19%)
9500/48617 (20%)
10000/48617 (21%)
10500/48617 (22%)
11000/48617 (23%)
11500/48617 (24%)
12000/48617 (25%)
12500/48617 (26%)
13000/48617 (27%)
13500/48617 (28%)
14000/48617 (29%)
14500/48617 (30%)
15000/48617 (31%)
15500/48617 (32%)
16000/48617 (33%)
16500/48617 (34%)
17000/48617 (35%)
17500/48617 (36%)
18000/48617 (37%)
18500/48617 (38%)
19000/48617 (39%)
19500/48617 (40%)
20000/48617 (41%)
20500/48617 (42%)
21000/48617 (43%)
21500/48617 (44%)
22000/48617 (45%)
22500/48617 (46%)
23000/48617 (47%)
23500/48617 (48%)
24000/48617 (49%)
24500/48617 (50%)
25000/48617 (51%)
25500/48617 (52%)
26000/48617 (53%)
26500/48617 (55%)
27000/48617 (56%)
27500/48617 (57%)
28000/48617 (58%)
28500/48617 (59%)
290

match_type
exact_raw                         48469
fuzzy_raw                            71
fuzzy_raw_coords_mismatch            60
exact_raw_coords_mismatch            12
unmatched                             3
fuzzy_translit_coords_mismatch        1
fuzzy_translit                        1
Name: count, dtype: int64

## Manual review helpers

Use these to sanity-check the weaker matches before trusting the output.

In [9]:
# Stops that found no acceptable candidate at all
df_result[df_result["match_type"] == "unmatched"][
    ["ontd_id", "ontd_name", "ontd_country"]
].shape

(3, 3)

In [10]:
# Weakest fuzzy matches — worth eyeballing first
df_result[df_result["match_type"].str.startswith("fuzzy")].sort_values(
    "name_score"
).shape

(133, 16)

In [11]:
# Same-name collisions resolved purely by distance/country (or not resolved at all)
df_result[df_result["candidate_count"] > 1].sort_values(
    "candidate_count", ascending=False
).shape

(4400, 16)